In [ ]:
import copy
import random
from agents import Agent, GraphicEnvironment, Direction, Bump, Thing

class Food(Thing):
    pass

class Water(Thing):
    pass

class SmartBlindDog(Agent):
    def __init__(self, program=None):
        super().__init__(program)
        self.location = [0, 0]
        self.direction = Direction("down")
        self.visited = set()
        self.visited.add((0, 0))

    def moveforward(self, success=True):
        if not success:
            return
        if self.direction.direction == Direction.R:
            self.location[0] += 1
        elif self.direction.direction == Direction.L:
            self.location[0] -= 1
        elif self.direction.direction == Direction.D:
            self.location[1] += 1
        elif self.direction.direction == Direction.U:
            self.location[1] -= 1
            
        self.visited.add(tuple(self.location))
        print(f"🚶 เดินมาที่พิกัด: {self.location}")

    def turn(self, d):
        self.direction = self.direction + d

    def eat(self, thing):
        if isinstance(thing, Food):
            print(f"SmartBlindDog: Ate Food at {self.location}")
            return True
        return False

    def drink(self, thing):
        if isinstance(thing, Water):
            print(f"SmartBlindDog: Drank Water at {self.location}")
            return True
        return False

def smart_dog_program(percepts):
    for p in percepts:
        if isinstance(p, Food):
            return 'eat'
        elif isinstance(p, Water):
            return 'drink'

    has_bump = any(isinstance(p, Bump) for p in percepts)
    
    agent = None
    for p in percepts:
        if isinstance(p, SmartBlindDog):
            agent = p
            break

    if agent is None:
        if has_bump:
            return random.choice(['turnright', 'turnleft'])
        return random.choice(['moveforward', 'turnright', 'turnleft'])

    def get_ahead_location(loc, direction_str):
        x, y = loc[0], loc[1]
        if direction_str == Direction.R: return (x + 1, y)
        if direction_str == Direction.L: return (x - 1, y)
        if direction_str == Direction.D: return (x, y + 1)
        if direction_str == Direction.U: return (x, y - 1)
        return (x, y)

    curr_dir = agent.direction.direction
    right_dir = (agent.direction + Direction.R).direction
    left_dir = (agent.direction + Direction.L).direction

    ahead_loc = get_ahead_location(agent.location, curr_dir)
    right_loc = get_ahead_location(agent.location, right_dir)
    left_loc = get_ahead_location(agent.location, left_dir)

    forward_is_unvisited = (not has_bump) and (ahead_loc not in agent.visited)
    right_is_unvisited = right_loc not in agent.visited
    left_is_unvisited = left_loc not in agent.visited

    if forward_is_unvisited:
        return 'moveforward'
    
    unvisited_turns = []
    if right_is_unvisited:
        unvisited_turns.append('turnright')
    if left_is_unvisited:
        unvisited_turns.append('turnleft')

    if unvisited_turns:
        return random.choice(unvisited_turns)

    if has_bump:
        return random.choice(['turnright', 'turnleft'])
    else:
        return random.choice(['moveforward', 'turnright', 'turnleft'])

class SmartPark2D(GraphicEnvironment):
    def percept(self, agent):
        things = self.list_things_at(agent.location)
        things.append(agent)
        
        loc = copy.deepcopy(agent.location)
        if agent.direction.direction == Direction.R: loc[0] += 1
        elif agent.direction.direction == Direction.L: loc[0] -= 1
        elif agent.direction.direction == Direction.D: loc[1] += 1
        elif agent.direction.direction == Direction.U: loc[1] -= 1
        if not (0 <= loc[0] < self.width and 0 <= loc[1] < self.height):
            things.append(Bump())
            
        return things
    
    def execute_action(self, agent, action):
        if action == 'turnright':
            agent.turn(Direction.R)
        elif action == 'turnleft':
            agent.turn(Direction.L)
        elif action == 'moveforward':
            agent.moveforward()
        elif action == "eat":
            items = self.list_things_at(agent.location, tclass=Food)
            if len(items) != 0:
                if agent.eat(items[0]):
                    self.delete_thing(items[0])
        elif action == "drink":
            items = self.list_things_at(agent.location, tclass=Water)
            if len(items) != 0:
                if agent.drink(items[0]):
                    self.delete_thing(items[0])

if __name__ == "__main__":
    park = SmartPark2D(5, 5, color={
        'SmartBlindDog': (200, 0, 0),
        'Water': (0, 200, 200),
        'Food': (230, 115, 40)
    })

    dog = SmartBlindDog(smart_dog_program)
    park.add_thing(dog, [0, 0])
    park.add_thing(Food(), [1, 2])
    park.add_thing(Water(), [0, 1])
    park.add_thing(Water(), [2, 4])
    park.add_thing(Food(), [4, 3])

    print("เริ่มการจำลอง SmartBlindDog")
    park.run(30, delay=1)
    
    print("สรุปพิกัดทั้งหมดที่ SmartBlindDog เคยสำรวจผ่าน:")
    print(sorted(list(dog.visited)))

,,,,
,,,,
,,,,
,,,,
,,,,


สรุปพิกัดทั้งหมดที่ SmartBlindDog เคยสำรวจผ่าน:
[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 0), (1, 1), (1, 2), (1, 4), (2, 0), (2, 4), (3, 0), (3, 4), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)]
